# Conditional CycleGAN — Results Visualisation

This notebook loads saved sample grids and the loss CSV to produce report figures.
It contains **no training logic, no model instantiation, and no dataloader code**.
Inference outputs are generated by calling `infer.py` via subprocess.

In [ ]:
import os
import subprocess
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import Image as IPImage, display

# ── Configuration ──────────────────────────────────────────────────────────
CHECKPOINT_DIR = 'checkpoints'
SAMPLE_DIR     = 'samples'
LOG_CSV        = 'logs/loss_log.csv'
RESULTS_DIR    = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Directories ready.')

## 1. Training Loss Curves

In [ ]:
df = pd.read_csv(LOG_CSV)
print(f'Loaded {len(df):,} log rows  |  epochs {df.epoch.min()}–{df.epoch.max()}')
df.head()

In [ ]:
# Average losses per epoch for clean curves
epoch_df = df.groupby('epoch').mean(numeric_only=True).reset_index()

loss_cols = ['loss_G', 'loss_D_A', 'loss_D_B',
             'loss_cycle', 'loss_identity', 'loss_cls', 'loss_perceptual']
titles    = ['Generator', 'Discriminator A', 'Discriminator B',
             'Cycle Consistency', 'Identity', 'Classification', 'Perceptual (VGG16)']

# Only plot columns that exist in this CSV (handles old + new log format)
available = [(c, t) for c, t in zip(loss_cols, titles) if c in epoch_df.columns]
n = len(available)
cols_per_row = 4
rows = (n + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(rows, cols_per_row, figsize=(5 * cols_per_row, 4 * rows))
axes_flat = axes.flat if hasattr(axes, 'flat') else [axes]

for ax, (col, title) in zip(axes_flat, available):
    ax.plot(epoch_df['epoch'], epoch_df[col], linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)

# Hide any unused subplots
for ax in list(axes_flat)[len(available):]:
    ax.set_visible(False)

plt.suptitle('Training Loss Curves (per-epoch average)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'loss_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR}/loss_curves.png")

## 2. Sample Grid Progression

In [ ]:
# Show sample grids at epochs 10, 50, 100, 150, 200
show_epochs = [10, 50, 100, 150, 200]

for ep in show_epochs:
    grid_path = os.path.join(SAMPLE_DIR, f'epoch_{ep:03d}.png')
    if os.path.exists(grid_path):
        print(f'\n── Epoch {ep} ──')
        display(IPImage(grid_path, width=800))
    else:
        print(f'[missing] {grid_path}')

## 3. Final Epoch Grid (Epoch 200)

In [ ]:
final_grid = os.path.join(SAMPLE_DIR, 'epoch_200.png')
if os.path.exists(final_grid):
    img = mpimg.imread(final_grid)
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Epoch 200 — Row 1: Real A  |  Row 2: Fake B  |  Row 3: Rec A',
                 fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'final_sample_grid.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'[missing] {final_grid} — run training first')

## 4. Inference via `infer.py`

Call the inference script via subprocess — keeps model instantiation out of this notebook.

In [ ]:
# ── Single-image inference example ──
# Edit paths to match your test image and desired output

CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'epoch_200.pt')
INPUT_IMG  = 'test_samples/player_01.png'   # must exist
TARGET_TEAM = '2'
NUM_TEAMS   = '4'
OUTPUT_IMG  = os.path.join(RESULTS_DIR, 'player_01_teamC.png')

if os.path.exists(CHECKPOINT) and os.path.exists(INPUT_IMG):
    result = subprocess.run(
        [
            'python', 'infer.py',
            '--checkpoint',  CHECKPOINT,
            '--input',       INPUT_IMG,
            '--target_team', TARGET_TEAM,
            '--num_teams',   NUM_TEAMS,
            '--output',      OUTPUT_IMG,
        ],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
    elif os.path.exists(OUTPUT_IMG):
        display(IPImage(OUTPUT_IMG, width=256))
else:
    missing = [p for p in [CHECKPOINT, INPUT_IMG] if not os.path.exists(p)]
    print(f'[skip] Missing files: {missing}')

## 5. Batch Inference

In [ ]:
BATCH_INPUT_DIR  = 'test_samples'
BATCH_OUTPUT_DIR = os.path.join(RESULTS_DIR, 'batch_teamC')

if os.path.exists(CHECKPOINT) and os.path.isdir(BATCH_INPUT_DIR):
    result = subprocess.run(
        [
            'python', 'infer.py',
            '--checkpoint',  CHECKPOINT,
            '--input_dir',   BATCH_INPUT_DIR,
            '--target_team', TARGET_TEAM,
            '--num_teams',   NUM_TEAMS,
            '--output_dir',  BATCH_OUTPUT_DIR,
        ],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
else:
    missing = [p for p in [CHECKPOINT, BATCH_INPUT_DIR] if not os.path.exists(p)]
    print(f'[skip] Missing: {missing}')

## 6. Available Checkpoints

In [ ]:
ckpts = sorted(Path(CHECKPOINT_DIR).glob('epoch_*.pt')) if Path(CHECKPOINT_DIR).exists() else []
print(f'Found {len(ckpts)} checkpoints:')
for c in ckpts:
    size_mb = c.stat().st_size / 1e6
    print(f'  {c.name}  ({size_mb:.1f} MB)')